In [1]:
import sys, os
# Path bootstrap to resolve imports correctly from src/ and project root
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import json
import pandas as pd
from types import SimpleNamespace

from src.architectures.agentic_guidelines.manager import DynamicGuidebookManager, DynamicGuidelinesManager
from src.architectures.agentic_guidelines.pipeline import AgenticNERPipeline as LSF_NER_Pipeline
from src.common.config import FORMATS
import src.common.brat2bio as brat2bio

brat2bio.options = SimpleNamespace(
    nosplit=True,
    annsuffix=".ann",
    singleclass=None
)


/Users/hmd/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
weave: wandb version 0.27.2 is available!  To upgrade, please run:
weave:  $ pip install wandb --upgrade
weave: weave version 0.52.43 is available!  To upgrade, please run:
weave:  $ pip install weave --upgrade
weave: Logged in as Weights & Biases user: hmdadham.
weave: View Weave data at https://wandb.ai/lifener/lsf-ner-tracing/weave
/Users/hmd/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# ==========================
# CONFIGURATION
# ==========================
FILE_FORMAT = "BIO"
MODEL_NAME = "deepseek-ai/deepseek-v4-pro"
BACKEND = "api"
DATA_FOLDER = os.path.join(PROJECT_ROOT, "data/600Abstracts/")
OUTPUT_BASE = os.path.join(PROJECT_ROOT, "output/agentic_guidelines")
PRED_FOLDER = os.path.join(OUTPUT_BASE, f"predicted/{FILE_FORMAT}/")
OUTPUT_FOLDER = PRED_FOLDER
BIO_GROUND_TRUTH = os.path.join(OUTPUT_BASE, "ground_truth_BIO/")
EVALUATION_FOLDER = os.path.join(OUTPUT_BASE, "combined_to_eval/")
DYNAMIC_GUIDEBOOK_PATH = os.path.join(PROJECT_ROOT, "data/guidelines/dynamicGuidebook.json")
DYNAMIC_GUIDELINES_PATH = DYNAMIC_GUIDEBOOK_PATH

os.makedirs(OUTPUT_FOLDER, exist_ok=True)
os.makedirs(PRED_FOLDER, exist_ok=True)
os.makedirs(BIO_GROUND_TRUTH, exist_ok=True)
os.makedirs(f"{BIO_GROUND_TRUTH}/train_phase", exist_ok=True)
os.makedirs(f"{BIO_GROUND_TRUTH}/test_phase", exist_ok=True)
os.makedirs(EVALUATION_FOLDER, exist_ok=True)

# ── Updated Imports (aligned with the new subpackage layout) ──
import os
import shutil
from src.architectures.agentic_guidelines.pipeline import AgenticNERPipeline as LSF_NER_Pipeline
from src.common.evaluation import evaluate_one_file, evaluate_all_files
from src.common.config import FORMATS
from src.common.bio2brat import BIO_to_standoff
import src.common.brat2bio as brat2bio
import json



# ==========================
# TRAINING PHASE
# ==========================
def train_pipeline(new_folder, train_folder, output_folder, model_name, backend="lmstudio", file_format=FILE_FORMAT):
    """
    Train the pipeline using .txt + .ann pairs and save the dynamicGuidelines externally.
    """
    if new_folder:
        os.makedirs(f"{output_folder}/{new_folder}", exist_ok=True)
        os.makedirs(os.path.join(EVALUATION_FOLDER, new_folder), exist_ok=True)
        os.makedirs(f"{BIO_GROUND_TRUTH}/train_phase/{new_folder}", exist_ok=True)
    
    pipeline = LSF_NER_Pipeline(model_name=model_name, backend=backend, file_format=file_format)
    train_files = sorted([f for f in os.listdir(train_folder) if f.endswith(".txt")])[0:19] # first 5 for training

    for file_name in train_files:
        base = os.path.splitext(file_name)[0]
        txt_path = os.path.join(train_folder, f"{base}.txt")
        ann_path = os.path.join(train_folder, f"{base}.ann")

        if not os.path.exists(ann_path):
            print(f"⚠️ Skipping {file_name} (missing .ann file)")
            continue

        with open(txt_path, "r", encoding="utf-8") as f:
            abstract = f.read().strip()

        if file_format == "BIO":
            # brat2bio.text_to_conll() uses both txt and ann files to produce BIO format
            with open(txt_path, "r", encoding="utf-8") as f:
                bio_output = brat2bio.text_to_conll(f)
            ground_truth = bio_output.getvalue()
            with open(os.path.join(BIO_GROUND_TRUTH, 'train_phase', new_folder, f'{base}.bio'), 'w') as f:
                f.write(ground_truth)
        else:
            with open(ann_path, "r", encoding="utf-8") as f:
                ground_truth = f.read()

        print("\n ---------------------------------------------------- \n")

        print(f"🧠 Training on: {file_name}")
        
        # print(f"ground : {ground_truth} \n \n abstract : {abstract}")

        predicted, reflection, dynamicGuidelines = pipeline.process_abstract(
            abstract,
            ground_truth=ground_truth,
            file_name=file_name 
        )
        if file_format == "BIO":
            shutil.copy(txt_path, f"{output_folder}/{new_folder}")
            copied_file_path = os.path.join(f"{output_folder}/{new_folder}", f"{base}.txt")
            with open(copied_file_path, "r", encoding="utf-8") as f:
                bio2ann = BIO_to_standoff(predicted.get("final_answer", "").strip(), abstract)
                with open(f"{output_folder}/{new_folder}/{base}.ann", "w", encoding="utf-8") as g:
                    for entity in bio2ann:
                        g.write(str(entity) + '\n')
                predicted_true_bio = brat2bio.text_to_conll(f)
            predicted_true_bio = predicted_true_bio.getvalue()

        output_ext = FORMATS[file_format]['File_format']
        output_path = os.path.join(f"{output_folder}/{new_folder}", f"{base}.{output_ext}")
        with open(output_path, "w", encoding="utf-8") as out_file:
            out_file.write(predicted_true_bio.strip() + "\n")
        
        # evaluate_one_file(base, "train_phase", mode=mode, folder_to_save=folder_to_save_eval)

        os.remove(copied_file_path)
        os.remove(f"{output_folder}/{new_folder}/{base}.ann")

    print(f"\n✅ Training complete. dynamicGuidelines saved to {DYNAMIC_GUIDELINES_PATH}")
    pipeline.dynamicGuidelines_manager.generate_curator_report()
    return train_files

    
# ==========================
# TESTING PHASE
# ==========================
def test_pipeline(new_folder, data_folder, pred_folder, model_name, backend="lmstudio", file_format=FILE_FORMAT):
    """
    Load dynamicGuidelines and run inference on unseen abstracts.
    """
    if new_folder:
        os.makedirs(f"{pred_folder}/{new_folder}", exist_ok=True)
        os.makedirs(os.path.join(EVALUATION_FOLDER, new_folder), exist_ok=True)
        os.makedirs(f"{BIO_GROUND_TRUTH}/test_phase/{new_folder}", exist_ok=True)

    pipeline = LSF_NER_Pipeline(model_name=model_name, backend=backend, file_format=file_format)

    all_txts = sorted([f for f in os.listdir(data_folder) if f.endswith(".txt")])
    test_files = all_txts[0:5]

    for file_name in test_files:
        base = os.path.splitext(file_name)[0]
        txt_path = os.path.join(data_folder, f"{base}.txt")
        ann_path = os.path.join(data_folder, f"{base}.ann")

        with open(txt_path, "r", encoding="utf-8") as f:
            abstract = f.read().strip()

        if file_format == "BIO":
            # brat2bio.text_to_conll() uses both txt and ann files to produce BIO format
            with open(txt_path, "r", encoding="utf-8") as f:
                bio_output = brat2bio.text_to_conll(f)
            ground_truth = bio_output.getvalue()
            x = f'test_phase/{new_folder}' if new_folder else 'test_phase'
            with open(os.path.join(BIO_GROUND_TRUTH, x, f'{base}.bio'), 'w') as f:
                f.write(ground_truth)
        else:
            with open(ann_path, "r", encoding="utf-8") as f:
                ground_truth = f.read()
        
        print(f"🔍 Testing on: {file_name}")
        predicted, reflection, _ = pipeline.process_abstract(
            abstract,
            ground_truth=ground_truth,
            file_name=file_name
        )

        if file_format=="BIO":
            shutil.copy(txt_path, f"{pred_folder}/{new_folder}")
            copied_file_path = os.path.join(f"{pred_folder}/{new_folder}", f"{base}.txt")
            with open(copied_file_path, "r", encoding="utf-8") as f:
                bio2ann = BIO_to_standoff(predicted.get("final_answer", "").strip(), abstract)
                with open(f"{pred_folder}/{new_folder}/{base}.ann", "w", encoding="utf-8") as g:
                    for entity in bio2ann:
                        g.write(str(entity) + '\n')
                predicted_true_bio = brat2bio.text_to_conll(f)
            predicted_true_bio = predicted_true_bio.getvalue()

        output_ext = FORMATS[file_format]['File_format']
        output_path = os.path.join(f"{pred_folder}/{new_folder}", f"{base}.{output_ext}")
        with open(output_path, "w", encoding="utf-8") as out_file:
            out_file.write(predicted_true_bio.strip() + "\n")
        
        # evaluate_one_file(base, "test_phase", mode=mode, folder_to_save=folder_to_save_eval)

        os.remove(copied_file_path)
        os.remove(f"{pred_folder}/{new_folder}/{base}.ann")

        print(f"💾 Saved prediction to: {output_path}")
        print("\n ---------------------------------------------------- \n")

    return test_files
    print("\n🎯 Testing complete — predictions saved to test folder.")


# ==========================
# MAIN EXECUTION
# ==========================
if __name__ == "__main__":
    print("🚀 Starting training phase...\n")
    # dynamicGuidelines_path = train_pipeline(
    #     train_folder=DATA_FOLDER,
    #     output_folder=OUTPUT_FOLDER,
    #     guideline=GUIDELINES,
    #     model_name=MODEL_NAME,
    #     backend=BACKEND,
    #     file_format=FILE_FORMAT,
    #     mode="exact",
    #     folder_to_save_eval=None
    # )

    # print("\n🚀 Starting testing phase...\n")
    # test_pipeline(
    #     data_folder=DATA_FOLDER,
    #     pred_folder=PRED_FOLDER,
    #     guideline=GUIDELINES,
    #     model_name=MODEL_NAME,
    #     backend=BACKEND,
    #     file_format=FILE_FORMAT,
    #     mode="overlap"
    # )
    # pass


🚀 Starting training phase...



In [4]:
train_flow = train_pipeline(
    new_folder="",
    train_folder="../data/Genereted_Subsets/subset_2_size_20",
    output_folder=OUTPUT_FOLDER,
    backend="api",
    file_format=FILE_FORMAT,
    model_name=MODEL_NAME
)
train_flow

Initialized SentenceTransformer on device: cpu
Loaded existing ChromaDB collection: dynamicGuidelines_bullets
Loaded dynamicGuidelines from /Users/hmd/Documents/GitHub/LSF-NER-Inital/output/DynamicGuideline/dynamicGuidelines.json
ChromaDB and dynamicGuidelines.json are in sync.

 ---------------------------------------------------- 

🧠 Training on: 11055166.txt
   📡 Calling API: NVIDIA (using key: nvapi-...6Fb-tr) with model: deepseek-ai/deepseek-v4-pro


Eliminate Textbound(start=811, end=820, type='Environmental_exposures', text='emissions in moving vehicles') due to overlap with Textbound(start=811, end=833, type='Environmental_exposures', text='emissions in buildings')
Eliminate Textbound(start=811, end=820, type='Environmental_exposures', text='emissions in moving vehicles') due to overlap with Textbound(start=811, end=833, type='Environmental_exposures', text='emissions in buildings')


   ⚠️ API call failed for NVIDIA (deepseek-ai/deepseek-v4-pro) on attempt 1/2: Request timed out.
      Retrying in 3 seconds...
   ⚠️ API call failed for NVIDIA (deepseek-ai/deepseek-v4-pro) on attempt 2/2: Request timed out.
   📡 Calling API: Cloudflare_AetherPool (using key: hmd) with model: deepseek-ai/deepseek-v4-pro
   ❌ Bad Request for Cloudflare_AetherPool (deepseek-ai/deepseek-v4-pro): Error code: 400 - {'error': {'message': "Model 'deepseek-ai/deepseek-v4-pro' is not supported. Choose a supported model or pass a direct Cloudflare tag starting with '@cf/'.", 'type': 'invalid_request_error', 'supported_models': ['deepseek-r1', 'gemma-4', 'gemma-3-12b', 'gpt-oss-120b', 'gpt-oss-20b', 'llama-4-scout', 'llama-3.1-8b', 'llama-3.1-70b', 'glm-4.7-flash', 'kimi-k2.6', 'qwen-3-30b']}}
   📡 Calling API: AvalAI (using key: aa-ZU6...muDsFs) with model: deepseek-ai/deepseek-v4-pro


KeyboardInterrupt: 

In [ ]:
test_flow = test_pipeline(
    new_folder="",
    data_folder="../data/Genereted_Subsets/temp",
    pred_folder="../output/temp",
    backend="api",
    file_format=FILE_FORMAT,
    model_name=MODEL_NAME
)
test_flow

In [ ]:
# from src.common.evaluation import evaluate_one_file

# evaluate_one_file(file_name=9678133, ground_truth_sub="test_phase/babak2", predicted_BIO_sub="babak2", folder_to_save="babak2")

In [ ]:
from src.common.weave_wrapper import LSFNEREval

# evaluate_one_file(10607166, "train_phase")
evaluate_one_file(28365427 , "train_phase")
# ----------------------------
# 1. Prepare file lists
# ----------------------------
# Single file for training
train_files = [f.replace(".txt", "") for f in train_flow]  # replace with your actual train files

# ----------------------------
# 2. Prepare DataFrames
# ----------------------------
train_df = pd.DataFrame({
    "file_name": train_files,
    "ground_truth_sub": ["train_phase"] * len(train_files)
})

# ----------------------------
# 3. Convert to Weave Datasets
# ----------------------------
train_dataset = weave.Dataset.from_pandas(train_df)

# ----------------------------
# 4. Define Weave Evaluations
# ----------------------------
train_eval = weave.Evaluation(
    name="LSF-NER-Train",
    dataset=train_dataset,
    scorers=[],  # direct use
    #scorers=[ner_scorer],  # direct use
)


# ----------------------------
# 5. Initialize Weave
# ----------------------------
weave.init("lsf-ner-tracing")  # replace with your project name

# ----------------------------
# 6. Run Evaluations (works in notebooks)
# ----------------------------
await train_eval.evaluate(LSFNEREval())



In [ ]:
from src.common.evaluation import evaluate_one_file, evaluate_all_files

# evaluate_one_file(10607166, "train_phase")
evaluate_one_file(28365427 , "train_phase")

In [ ]:
FILE_FORMAT = "BIO"
OUTPUT_BASE = os.path.join(PROJECT_ROOT, "output/agentic_guidelines")
PRED_FOLDER = os.path.join(OUTPUT_BASE, f"predicted/{FILE_FORMAT}/")
PRED_FOLDER = os.path.join(OUTPUT_BASE, "temp/")

evaluate_all_files(PRED_FOLDER, "test_phase", mode="exact")